# Silver Layer — Cleaned & Enriched
Read Bronze Parquet → parse timestamp → left-join hostname dim → flag bots → write Silver Parquet.
No rows dropped (even bots stay — Gold layer will split them).

In [1]:
import os, sys
os.environ['SPARK_HOME'] = '/usr/local/spark-3.5.0-bin-hadoop3'
os.environ['HADOOP_USER_NAME'] = 'root'
sys.path.insert(0, '/usr/local/spark-3.5.0-bin-hadoop3/python')
sys.path.insert(0, '/usr/local/spark-3.5.0-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip')
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('03_silver')
    .master('local[4]')
    .config('spark.hadoop.fs.defaultFS', 'hdfs://hdfs-namenode:9000')
    .config('spark.sql.shuffle.partitions', '16')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .config('spark.sql.broadcastTimeout', '300')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)


Spark version: 3.5.0


In [2]:
# Read Bronze
bronze = spark.read.parquet('hdfs://hdfs-namenode:9000/data/bronze/access_logs')
print(f'Bronze rows: {bronze.count():,}')
bronze.printSchema()


Bronze rows: 10,365,077
root
 |-- ip: string (nullable = true)
 |-- ts_raw: string (nullable = true)
 |-- method: string (nullable = true)
 |-- path: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- bytes: long (nullable = true)
 |-- referrer: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- xff: string (nullable = true)
 |-- log_date: date (nullable = true)



In [3]:
# Read hostname dimension — 12.8 MB, safe to broadcast
dim = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'false')
    .csv('hdfs://hdfs-namenode:9000/raw/client-hostname/client_hostname.csv')
    .select(
        F.col('client').alias('ip'),
        F.col('hostname')
    )
    # Where hostname == IP it means DNS resolution failed; use NULL instead
    .withColumn('hostname',
        F.when(F.col('hostname') == F.col('ip'), None)
         .otherwise(F.col('hostname'))
    )
)
print(f'Dim rows: {dim.count():,}')
dim.show(3, truncate=60)


Dim rows: 258,445
+------------+--------+
|          ip|hostname|
+------------+--------+
|5.123.144.95|    NULL|
|5.122.76.187|    NULL|
|5.215.249.99|    NULL|
+------------+--------+
only showing top 3 rows



In [4]:
# Bot keywords identified from Phase 1 real UA analysis
BOT_RE = r'(?i)(bot|spider|crawl|ahref|python-requests|bingpreview|slurp|semrush|dataprovider|zgrab)'

silver = (
    bronze
    # 1. Parse timestamp to proper TimestampType
    #    Format: '22/Jan/2019:03:56:14 +0330'
    .withColumn(
        'ts',
        F.to_timestamp('ts_raw', 'dd/MMM/yyyy:HH:mm:ss Z')
    )
    .withColumn('hour', F.hour('ts'))
    # 2. Left-join hostname dimension (broadcast — 12.8 MB)
    .join(F.broadcast(dim), on='ip', how='left')
    # If no hostname resolved, fall back to raw IP
    .withColumn('hostname',
        F.coalesce(F.col('hostname'), F.col('ip'))
    )
    # 3. Bot flag from real UA list
    .withColumn('is_bot',
        F.col('user_agent').rlike(BOT_RE)
    )
    # 4. Error flag
    .withColumn('is_error', F.col('status') >= 400)
    # 5. Clean up columns — drop raw ts_raw
    .drop('ts_raw')
    .select(
        'ip', 'hostname', 'ts', 'log_date', 'hour',
        'method', 'path', 'protocol', 'status', 'bytes',
        'referrer', 'user_agent', 'xff',
        'is_bot', 'is_error'
    )
)

silver.printSchema()
silver.show(3, truncate=80)


root
 |-- ip: string (nullable = true)
 |-- hostname: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- log_date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- method: string (nullable = true)
 |-- path: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- bytes: long (nullable = true)
 |-- referrer: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- xff: string (nullable = true)
 |-- is_bot: boolean (nullable = true)
 |-- is_error: boolean (nullable = true)

+-------------+-------------+-------------------+----------+----+------+-----------------------------------+--------+------+-----+----------------------------------------------------+--------------------------------------------------------------------------------+---+------+--------+
|           ip|     hostname|                 ts|  log_date|hour|method|                               path|protocol|status|bytes|        

In [5]:
# Write Silver Parquet to HDFS, partitioned by log_date
silver.write.mode('overwrite').partitionBy('log_date').parquet(
    'hdfs://hdfs-namenode:9000/data/silver/access_logs'
)
print('Silver write complete.')

# Read back and verify
verify = spark.read.parquet('hdfs://hdfs-namenode:9000/data/silver/access_logs')
row_count = verify.count()
print(f'Silver row count (from Parquet): {row_count:,}')

# Quick stats
bot_count   = verify.filter(F.col('is_bot')).count()
error_count = verify.filter(F.col('is_error')).count()
print(f'Bot requests    : {bot_count:,}  ({100*bot_count/row_count:.1f}%)')
print(f'Error requests  : {error_count:,}  ({100*error_count/row_count:.1f}%)')
verify.select('log_date').distinct().orderBy('log_date').show()


Silver write complete.
Silver row count (from Parquet): 10,365,077
Bot requests    : 1,131,307  (10.9%)
Error requests  : 177,634  (1.7%)
+----------+
|  log_date|
+----------+
|2019-01-22|
|2019-01-23|
|2019-01-24|
|2019-01-25|
|2019-01-26|
+----------+



In [6]:
spark.stop()
print('Silver layer done.')


Silver layer done.
